# Agentic Alpha Model Replay Search V1

Diagnostic R-G-E-V-D continuation card for SOXL model replay, qlib daily LGBM selector, and cross-asset qlib selector branches.

In [1]:

from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import qlib
from lightgbm import LGBMClassifier
from qlib.data import D
from qlib.workflow import R

ROOT_CANDIDATES = [Path.cwd(), Path('/Users/rziiiii/dev/workspace')]
ROOT = next(
    path
    for base in ROOT_CANDIDATES
    for path in [base, *base.parents]
    if (path / 'AGENTS.md').exists() and (path / 'SPEC.md').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.workflow import load_agentic_alpha_backtest_metric

EXPERIMENT = 'agentic_alpha_model_replay_search_v1'
CARD = ROOT / 'research' / EXPERIMENT
RECORDER_URI = (CARD / 'qlib_recorder').resolve()
SOXL_PROVIDER = (ROOT / 'data' / 'qlib' / 'soxl').resolve()
START = '2019-01-02'
END = '2026-06-09'
CROSS_END = '2026-04-10'
MAX_DRAWDOWN_FLOOR = -20.0
TRANSACTION_COST = 0.0003

CARD.mkdir(parents=True, exist_ok=True)
if RECORDER_URI.exists():
    shutil.rmtree(RECORDER_URI)
RECORDER_URI.mkdir(parents=True, exist_ok=True)

workflow_yaml = f'''experiment: {EXPERIMENT}
card_type: diagnostic_search
qlib:
  provider_uri: {SOXL_PROVIDER.as_posix()}
  recorder_uri: {RECORDER_URI.as_posix()}
  market: all
  region: us
  auxiliary_providers:
    - data/qlib/tqqq
    - data/qlib/tecl
    - data/qlib/spxl
    - data/qlib/upro
segments:
  test:
    start: {START}
    end: {END}
  cross_asset_test:
    start: {START}
    end: {CROSS_END}
rgevd:
  branch: model_replay_and_cross_asset_selector
  stages:
    - research
    - retrieve
    - generate
    - evaluate
    - validate
    - distill
goal_gate:
  baseline_card: production/qlib_paper_sim_v1
  max_drawdown_floor: {MAX_DRAWDOWN_FLOOR}
  require_return_above_baseline: true
artifact_policy:
  policy: slim_card_recorder_first
  machine_entrypoint: summary.json
'''
(CARD / 'workflow.yaml').write_text(workflow_yaml, encoding='utf-8')

baseline = load_agentic_alpha_backtest_metric(ROOT / 'production' / 'qlib_paper_sim_v1', window='test')
production_summary = json.loads((ROOT / 'production' / 'qlib_paper_sim_v1' / 'summary.json').read_text(encoding='utf-8'))
production_recorder = Path(production_summary['recorder_local_dir'])

external_sources = pd.DataFrame(
    [
        {
            'source_type': 'github',
            'title': 'agentic-alpha R-G-E-V-D workflow',
            'url': 'https://github.com/JacobDu/agentic-alpha',
            'extracted_signal': 'Use retrieve-generate-evaluate-validate-distill loops to close failed strategy branches with evidence.',
            'retrieval_status': 'reachable',
        },
        {
            'source_type': 'arxiv',
            'title': 'Compounding Effects in Leveraged ETFs',
            'url': 'https://arxiv.org/html/2504.20116v1',
            'extracted_signal': 'LETF compounding depends on autocorrelation, volatility clustering, trend, and mean reversion regimes.',
            'retrieval_status': 'reachable',
        },
        {
            'source_type': 'arxiv',
            'title': 'Single-Asset Adaptive Leveraged Volatility Control',
            'url': 'https://arxiv.org/html/2603.01298v2',
            'extracted_signal': 'Feedback volatility control motivates target volatility, exposure clipping, smoothing, and leverage bounds.',
            'retrieval_status': 'reachable',
        },
        {
            'source_type': 'paper',
            'title': 'Volatility Targeting Is Trendy',
            'url': 'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=4773781',
            'extracted_signal': 'Volatility-managed alpha can overlap with trend-following exposure, so trend gates must be tested directly.',
            'retrieval_status': 'reachable',
        },
        {
            'source_type': 'github',
            'title': 'MLM trend-following implementation',
            'url': 'https://github.com/amstrdm/mlm-trend-following',
            'extracted_signal': 'A 200-day trend filter and volatility filter are a concrete external implementation pattern to test.',
            'retrieval_status': 'reachable',
        },
        {
            'source_type': 'twitter',
            'title': 'SOXL realized versus implied volatility ratio discussion',
            'url': 'https://x.com/Alpha_Ex_LLC/status/2067644619462938886',
            'extracted_signal': 'SOXL realized/implied volatility ratio discussion motivates testing volatility-ratio and VIX regime guards.',
            'retrieval_status': 'search_snippet_only',
        },
    ]
)
external_sources.to_csv(CARD / 'external_sources.csv', index=False)


def init_provider(provider: Path) -> None:
    qlib.init(provider_uri=provider.resolve().as_posix(), region='us', kernels=1, clear_mem_cache=True)


def max_drawdown_pct(nav: pd.Series | np.ndarray) -> float:
    nav = pd.Series(nav).astype(float)
    return float((nav / nav.cummax() - 1.0).min() * 100.0)


def vector_metrics(index: pd.Index, weight: np.ndarray, forward_return: np.ndarray) -> dict[str, float]:
    weight = np.nan_to_num(weight, nan=0.0, posinf=0.0, neginf=0.0)
    weight = np.clip(weight, 0.0, 10.0)
    turnover_path = np.abs(np.diff(np.r_[0.0, weight]))
    strategy_return = weight * forward_return - turnover_path * TRANSACTION_COST
    nav = pd.Series(np.cumprod(1.0 + strategy_return), index=index)
    return {
        'return_pct': float((nav.iloc[-1] - 1.0) * 100.0),
        'max_drawdown_pct': max_drawdown_pct(nav),
        'turnover': float(turnover_path.sum()),
        'mean_exposure_pct': float(weight.mean() * 100.0),
    }


def branch_rows(branch: str, grid: pd.DataFrame, *, coverage_start: str, coverage_end: str, comparison_return: float) -> tuple[list[dict], dict]:
    grid = grid.copy()
    grid['beats_goal_baseline'] = grid['return_pct'] > baseline.return_pct
    grid['beats_comparison_baseline'] = grid['return_pct'] > comparison_return
    grid['passes_drawdown_gate'] = grid['max_drawdown_pct'] >= MAX_DRAWDOWN_FLOOR
    passes_both = grid[grid['beats_goal_baseline'] & grid['passes_drawdown_gate']]
    best_return = grid.sort_values('return_pct', ascending=False).iloc[0]
    drawdown_pool = grid[grid['passes_drawdown_gate']]
    best_drawdown = drawdown_pool.sort_values('return_pct', ascending=False).iloc[0] if not drawdown_pool.empty else grid.sort_values('max_drawdown_pct', ascending=False).iloc[0]
    rows = []
    for label, row in [('best_return', best_return), ('best_drawdown_gate', best_drawdown)]:
        decision = 'Promote' if bool(row['beats_goal_baseline']) and bool(row['passes_drawdown_gate']) else ('Drop' if not bool(row['passes_drawdown_gate']) else 'Iterate')
        decision_basis = 'beats_goal_baseline_and_drawdown_gate' if decision == 'Promote' else ('drawdown_gate' if not bool(row['passes_drawdown_gate']) else 'return_gate')
        rows.append(
            {
                'branch': branch,
                'candidate': f'{branch}_{label}',
                'coverage_start': coverage_start,
                'coverage_end': coverage_end,
                'return_pct': float(row['return_pct']),
                'max_drawdown_pct': float(row['max_drawdown_pct']),
                'baseline_return_pct': float(baseline.return_pct),
                'comparison_baseline_return_pct': float(comparison_return),
                'max_drawdown_floor': MAX_DRAWDOWN_FLOOR,
                'beats_goal_baseline': bool(row['beats_goal_baseline']),
                'passes_drawdown_gate': bool(row['passes_drawdown_gate']),
                'decision': decision,
                'decision_basis': decision_basis,
                'params': json.dumps(
                    {
                        key: (None if pd.isna(row[key]) else row[key].item() if hasattr(row[key], 'item') else row[key])
                        for key in grid.columns
                        if key not in {'return_pct', 'max_drawdown_pct', 'turnover', 'mean_exposure_pct', 'beats_goal_baseline', 'beats_comparison_baseline', 'passes_drawdown_gate'}
                    },
                    sort_keys=True,
                ),
            }
        )
    summary = {
        'coverage_start': coverage_start,
        'coverage_end': coverage_end,
        'grid_rows': int(len(grid)),
        'passes_both_count': int(len(passes_both)),
        'comparison_baseline_return_pct': float(comparison_return),
        'goal_baseline_return_pct': float(baseline.return_pct),
        'best_return_pct': float(best_return['return_pct']),
        'best_return_max_drawdown_pct': float(best_return['max_drawdown_pct']),
        'best_drawdown_gate_return_pct': float(best_drawdown['return_pct']),
        'best_drawdown_gate_max_drawdown_pct': float(best_drawdown['max_drawdown_pct']),
    }
    return rows, summary


def production_signal_overlay_branch() -> tuple[list[dict], dict]:
    pred = pd.read_pickle(production_recorder / 'artifacts' / 'pred.pkl').reset_index().rename(columns={'score': 'score'})
    label = pd.read_pickle(production_recorder / 'artifacts' / 'label.pkl').reset_index()
    label_col = [col for col in label.columns if col not in {'datetime', 'instrument'}][0]
    label = label.rename(columns={label_col: 'fwd_ret'})
    init_provider(SOXL_PROVIDER)
    features = D.features(['SOXL', 'SOXX'], ['$close', '$vix'], start_time='2018-01-01', end_time='2026-06-10').reset_index()
    soxl = features[features['instrument'] == 'SOXL'][['datetime', '$close', '$vix']].rename(columns={'$close': 'soxl_close', '$vix': 'vix'})
    soxx = features[features['instrument'] == 'SOXX'][['datetime', '$close']].rename(columns={'$close': 'soxx_close'})
    frame = pred.merge(label[['datetime', 'instrument', 'fwd_ret']], on=['datetime', 'instrument'], how='left').merge(soxl, on='datetime').merge(soxx, on='datetime').sort_values('datetime').reset_index(drop=True)
    frame['realized_vol_20'] = frame['fwd_ret'].rolling(20).std().shift(1) * np.sqrt(252)
    frame['sma_50'] = frame['soxx_close'].rolling(50).mean().shift(1)
    frame['sma_200'] = frame['soxx_close'].rolling(200).mean().shift(1)
    trend = (frame['sma_50'] > frame['sma_200']).fillna(False).to_numpy(bool)
    ret = frame['fwd_ret'].fillna(0.0).to_numpy(float)
    vol = frame['realized_vol_20'].to_numpy(float)
    vix = frame['vix'].to_numpy(float)
    score = frame['score'].to_numpy(float)
    rows = []
    for q in np.linspace(0.05, 0.50, 10):
        signal = (score > np.nanquantile(score, q)).astype(float) * trend.astype(float)
        for target_vol in [None, 0.25, 0.30, 0.40, 0.60, 0.80, 1.00]:
            base_weight = signal.copy()
            if target_vol is not None:
                base_weight = base_weight * np.clip(target_vol / vol, 0.0, 1.0)
            for vix_max in [None, 25, 30, 35, 40, 50]:
                weight = base_weight if vix_max is None else np.where(vix <= vix_max, base_weight, 0.0)
                metrics = vector_metrics(pd.RangeIndex(len(frame)), weight, ret)
                rows.append({'score_quantile': q, 'target_vol': target_vol, 'vix_max': vix_max, **metrics})
    grid = pd.DataFrame(rows)
    return branch_rows('production_signal_overlay', grid, coverage_start=START, coverage_end=END, comparison_return=baseline.return_pct)


def add_technical_features(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    close_cols = [col for col in frame.columns if col.endswith('_close')]
    for close_col in close_cols:
        prefix = close_col[:-6]
        close = frame[close_col]
        frame[f'{prefix}_ret1'] = close.pct_change(1)
        for n in [2, 3, 5, 10, 21, 42, 63, 126, 252]:
            frame[f'{prefix}_ret{n}'] = close.pct_change(n)
        for n in [5, 10, 20, 42, 63]:
            frame[f'{prefix}_vol{n}'] = frame[f'{prefix}_ret1'].rolling(n).std() * np.sqrt(252)
            frame[f'{prefix}_ma_ratio{n}'] = close / close.rolling(n).mean() - 1.0
            frame[f'{prefix}_hh_dist{n}'] = close / close.rolling(n).max() - 1.0
    frame['vix'] = frame['SOXL_vix']
    for n in [3, 5, 10, 21, 42, 63]:
        frame[f'vix_chg{n}'] = frame['vix'].pct_change(n)
        frame[f'vix_ma_ratio{n}'] = frame['vix'] / frame['vix'].rolling(n).mean() - 1.0
    frame['fwd_ret'] = frame['SOXL_close'].pct_change().shift(-1)
    frame['target_up'] = (frame['fwd_ret'] > 0).astype(int)
    return frame.replace([np.inf, -np.inf], np.nan)


def build_daily_feature_frame(provider: Path, start: str, end: str) -> pd.DataFrame:
    init_provider(provider)
    raw = D.features(['SOXL', 'SOXX'], ['$open', '$high', '$low', '$close', '$volume', '$vix'], start_time=start, end_time=end).reset_index()
    wide = {}
    for inst in ['SOXL', 'SOXX']:
        sub = raw[raw['instrument'] == inst].sort_values('datetime').set_index('datetime')
        for col in ['$open', '$high', '$low', '$close', '$volume', '$vix']:
            wide[f'{inst}_{col[1:]}'] = sub[col]
    return add_technical_features(pd.DataFrame(wide).sort_index())


def feature_columns(frame: pd.DataFrame) -> list[str]:
    return [
        col
        for col in frame.columns
        if col not in {'fwd_ret', 'target_up', 'SOXX_vix'}
        and not col.endswith(('_open', '_high', '_low', '_close', '_volume', '_vix'))
        and not frame[col].isna().all()
    ]


def walk_forward_lgbm_predictions(frame: pd.DataFrame, features: list[str], *, test_start: str, test_end: str) -> pd.Series:
    clean = frame.dropna(subset=features + ['fwd_ret', 'target_up']).copy()
    test_dates = clean.loc[(clean.index >= pd.Timestamp(test_start)) & (clean.index <= pd.Timestamp(test_end))].index
    model = LGBMClassifier(
        n_estimators=90,
        learning_rate=0.025,
        max_depth=3,
        num_leaves=7,
        min_child_samples=35,
        subsample=0.8,
        colsample_bytree=0.75,
        reg_lambda=3.0,
        random_state=11,
        verbose=-1,
    )
    outputs = []
    for start_i in range(0, len(test_dates), 126):
        dates = test_dates[start_i : start_i + 126]
        train = clean.loc[(clean.index >= pd.Timestamp('2012-01-03')) & (clean.index < dates[0])]
        test = clean.loc[dates]
        model.fit(train[features], train['target_up'])
        outputs.append(pd.Series(model.predict_proba(test[features])[:, 1], index=test.index, name='score'))
    return pd.concat(outputs)


def lgbm_grid_branch(branch: str, frame: pd.DataFrame, pred: pd.Series, *, coverage_end: str, comparison_return: float) -> tuple[list[dict], dict]:
    base = frame.loc[pred.index, ['fwd_ret', 'vix', 'SOXX_ret63', 'SOXX_ma_ratio63', 'SOXL_vol20']].dropna()
    pred = pred.loc[base.index]
    ret = base['fwd_ret'].to_numpy(float)
    vol = base['SOXL_vol20'].to_numpy(float)
    vix = base['vix'].to_numpy(float)
    trend_gate = ((base['SOXX_ret63'] > 0) & (base['SOXX_ma_ratio63'] > -0.03)).to_numpy(bool)
    score = pred.to_numpy(float)
    rows = []
    for threshold in np.quantile(score, np.linspace(0.10, 0.80, 8)):
        signal = (score > threshold).astype(float)
        for cap in [0.75, 1.0, 1.25, 1.5]:
            for target_vol in [0.20, 0.25, 0.30, 0.40, None]:
                weight = signal * cap
                if target_vol is not None:
                    weight = weight * np.clip(target_vol / vol, 0.0, cap)
                for vix_max in [None, 25, 30, 40, 50]:
                    gated = weight if vix_max is None else np.where(vix <= vix_max, weight, 0.0)
                    for use_trend_gate in [False, True]:
                        final_weight = np.where(trend_gate, gated, 0.0) if use_trend_gate else gated
                        metrics = vector_metrics(base.index, final_weight, ret)
                        rows.append({'threshold': float(threshold), 'cap': cap, 'target_vol': target_vol, 'vix_max': vix_max, 'trend_gate': use_trend_gate, **metrics})
    grid = pd.DataFrame(rows)
    return branch_rows(branch, grid, coverage_start=START, coverage_end=coverage_end, comparison_return=comparison_return)


def qlib_daily_lgbm_branch() -> tuple[list[dict], dict]:
    frame = build_daily_feature_frame(SOXL_PROVIDER, '2010-03-11', '2026-06-10')
    features = feature_columns(frame)
    pred = walk_forward_lgbm_predictions(frame, features, test_start=START, test_end=END)
    return lgbm_grid_branch('qlib_daily_lgbm_replay', frame, pred, coverage_end=END, comparison_return=baseline.return_pct)


def cross_asset_lgbm_branch() -> tuple[list[dict], dict]:
    providers = {
        'soxl': ['SOXL', 'SOXX'],
        'tqqq': ['TQQQ', 'QQQ'],
        'tecl': ['TECL', 'XLK'],
        'spxl': ['SPXL', 'SPY'],
        'upro': ['UPRO', 'SPY'],
    }
    frames = []
    for store, instruments in providers.items():
        provider = ROOT / 'data' / 'qlib' / store
        init_provider(provider)
        raw = D.features(instruments, ['$open', '$high', '$low', '$close', '$volume', '$vix'], start_time='2010-03-11', end_time='2026-04-13').reset_index()
        for inst in instruments:
            sub = raw[raw['instrument'] == inst].sort_values('datetime').set_index('datetime')
            prefix = inst if inst != 'SPY' else f'{store.upper()}_SPY'
            frames.append(pd.DataFrame({f'{prefix}_{col[1:]}': sub[col] for col in ['$open', '$high', '$low', '$close', '$volume', '$vix']}))
    frame = pd.concat(frames, axis=1).sort_index()
    frame = frame.loc[:, ~frame.columns.duplicated()]
    frame = add_technical_features(frame)
    features = feature_columns(frame)
    pred = walk_forward_lgbm_predictions(frame, features, test_start=START, test_end=CROSS_END)
    report = pd.read_pickle(production_recorder / 'artifacts' / 'portfolio_analysis' / 'report_normal_1day.pkl')
    report = report.loc[(report.index >= pd.Timestamp(START)) & (report.index <= pd.Timestamp(CROSS_END))]
    nav = (1.0 + report['return'].astype(float) - report['cost'].astype(float)).cumprod()
    comparison_return = float((nav.iloc[-1] - 1.0) * 100.0)
    return lgbm_grid_branch('cross_asset_lgbm_replay', frame, pred, coverage_end=CROSS_END, comparison_return=comparison_return)

all_rows = []
branch_results = {}
for rows, branch_summary in [production_signal_overlay_branch(), qlib_daily_lgbm_branch(), cross_asset_lgbm_branch()]:
    all_rows.extend(rows)
    branch_results[rows[0]['branch']] = branch_summary

candidate_metrics = pd.DataFrame(all_rows)
candidate_metrics.to_csv(CARD / 'agentic_alpha_model_replay_candidate_metrics.csv', index=False)
passes_both_count = int(((candidate_metrics['return_pct'] > baseline.return_pct) & (candidate_metrics['max_drawdown_pct'] >= MAX_DRAWDOWN_FLOOR)).sum())
best_candidate = None
if passes_both_count:
    best_candidate = str(candidate_metrics[(candidate_metrics['return_pct'] > baseline.return_pct) & (candidate_metrics['max_drawdown_pct'] >= MAX_DRAWDOWN_FLOOR)].sort_values('return_pct', ascending=False).iloc[0]['candidate'])

search_summary = {
    'workflow': 'agentic_alpha_model_replay_search',
    'stage_order': ['research', 'retrieve', 'generate', 'evaluate', 'validate', 'distill'],
    'goal_achieved': bool(best_candidate),
    'best_candidate': best_candidate,
    'passes_both_count': passes_both_count,
    'baseline': {
        'name': baseline.name,
        'return_pct': float(baseline.return_pct),
        'max_drawdown_pct': float(baseline.max_drawdown_pct),
        'benchmark_return_pct': float(baseline.benchmark_return_pct),
        'recorder_id': baseline.recorder_id,
    },
    'branch_results': branch_results,
    'external_source_counts': external_sources['source_type'].value_counts().sort_index().to_dict(),
}
(CARD / 'agentic_alpha_model_replay_search_summary.json').write_text(json.dumps(search_summary, indent=2, sort_keys=True), encoding='utf-8')
(CARD / 'failure_modes.md').write_text(
    '# Failure Modes\n\n'
    '- production_signal_overlay: existing production signal can exceed the baseline return only while retaining deep drawdowns, so the drawdown gate fails.\n'
    '- qlib_daily_lgbm_replay: walk-forward daily LGBM selectors that pass MaxDD < 20% retain too little exposure to beat the production baseline.\n'
    '- cross_asset_lgbm_replay: auxiliary qlib stores add TQQQ/QQQ, TECL/XLK, SPXL/SPY, and UPRO/SPY context but only cover through 2026-04-10 for the aligned forward-return test, so it cannot prove the full goal and still fails the return gate.\n',
    encoding='utf-8',
)

R.set_uri(RECORDER_URI.as_posix())
with R.start(experiment_name=EXPERIMENT):
    recorder = R.get_recorder()
    recorder_id = recorder.id
    R.log_params(
        baseline=baseline.name,
        max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
        branch_count=len(branch_results),
        candidate_count=len(candidate_metrics),
        provider_uri=SOXL_PROVIDER.as_posix(),
    )
    for path in [
        CARD / 'agentic_alpha_model_replay_candidate_metrics.csv',
        CARD / 'agentic_alpha_model_replay_search_summary.json',
        CARD / 'external_sources.csv',
        CARD / 'failure_modes.md',
    ]:
        recorder.log_artifact(path.as_posix(), artifact_path='agentic_alpha_model_replay')
    recorder_local_dir = Path(recorder.get_local_dir())

for transient_name in ['code_cached.txt', 'code_diff.txt', 'code_status.txt']:
    transient_path = recorder_local_dir / 'artifacts' / transient_name
    if transient_path.exists():
        transient_path.unlink()

recorder_artifacts = sorted(path.relative_to(recorder_local_dir).as_posix() for path in recorder_local_dir.rglob('*') if path.is_file())
search_summary['recorder_id'] = recorder_id
search_summary['recorder_artifacts'] = recorder_artifacts
(CARD / 'agentic_alpha_model_replay_search_summary.json').write_text(json.dumps(search_summary, indent=2, sort_keys=True), encoding='utf-8')

summary = {
    'experiment': EXPERIMENT,
    'card_type': 'diagnostic_search',
    'workflow_config': 'workflow.yaml',
    'provider_uri': SOXL_PROVIDER.as_posix(),
    'auxiliary_provider_uris': {
        'tqqq': (ROOT / 'data' / 'qlib' / 'tqqq').resolve().as_posix(),
        'tecl': (ROOT / 'data' / 'qlib' / 'tecl').resolve().as_posix(),
        'spxl': (ROOT / 'data' / 'qlib' / 'spxl').resolve().as_posix(),
        'upro': (ROOT / 'data' / 'qlib' / 'upro').resolve().as_posix(),
    },
    'recorder_uri': RECORDER_URI.as_posix(),
    'recorder_id': recorder_id,
    'recorder_local_dir': recorder_local_dir.as_posix(),
    'recorder_artifacts': recorder_artifacts,
    'market': 'all',
    'region': 'us',
    'as_of': END,
    'data_range': [START, END],
    'qlib_native_api': ['qlib.init', 'qlib.data.D.features', 'qlib.workflow.R.start', 'clear_mem_cache=True', 'LGBMClassifier'],
    'qlib_docs': {
        'data': 'https://qlib.readthedocs.io/en/latest/component/data.html',
        'model': 'https://qlib.readthedocs.io/en/latest/component/model.html',
        'workflow': 'https://qlib.readthedocs.io/en/latest/component/workflow.html',
    },
    'baseline': search_summary['baseline'],
    'goal_gate': {'max_drawdown_floor': MAX_DRAWDOWN_FLOOR, 'require_return_above_baseline': True},
    'branch_results': branch_results,
    'search_report': search_summary,
    'promotion_decision': 'not_promoted' if not search_summary['goal_achieved'] else 'promote_candidate',
    'sidecars': {
        'candidate_metrics': 'agentic_alpha_model_replay_candidate_metrics.csv',
        'external_sources': 'external_sources.csv',
        'failure_modes': 'failure_modes.md',
        'search_summary': 'agentic_alpha_model_replay_search_summary.json',
    },
}
summary['artifact_policy'] = {
    'policy': 'slim_card_recorder_first',
    'primary_artifact': f'{EXPERIMENT}.ipynb',
    'machine_entrypoint': 'summary.json',
    'workflow_config': 'workflow.yaml',
    'kept_sidecars': sorted(summary['sidecars'].values()),
    'reproducible_from_recorder': True,
    'recorder_only_artifacts': {
        'predictions': 'artifacts/pred.pkl',
        'labels': 'artifacts/label.pkl',
        'all_model_predictions': 'artifacts/model_selection/all_model_predictions.csv',
        'signals': 'artifacts/signals.pkl',
        'portfolio_report': 'artifacts/portfolio_analysis/report_normal_1day.pkl',
        'positions': 'artifacts/portfolio_analysis/positions_normal_1day.pkl',
        'risk_analysis': 'artifacts/portfolio_analysis/port_analysis_1day.pkl',
    },
}
(CARD / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')
conclusion = f'''# 中文结论

- 做了什么：继续 R-G-E-V-D 的下一轮，把外部 LETF/vol-control/trend-following 知识转成 model replay、日线 LGBM selector 和跨资产 LGBM selector 三个候选分支，并通过 qlib Data API 与生产 recorder 回放验证。
- 核心指标：生产基准 {baseline.name} full-test return_pct={baseline.return_pct:.4f}、max_drawdown_pct={baseline.max_drawdown_pct:.4f}；本卡三个分支 passes_both_count={passes_both_count}。
- 分支结果：production_signal_overlay 最高收益候选 return_pct={branch_results['production_signal_overlay']['best_return_pct']:.4f} 但 max_drawdown_pct={branch_results['production_signal_overlay']['best_return_max_drawdown_pct']:.4f}；qlib_daily_lgbm_replay 最佳 20% 回撤候选 return_pct={branch_results['qlib_daily_lgbm_replay']['best_drawdown_gate_return_pct']:.4f}、max_drawdown_pct={branch_results['qlib_daily_lgbm_replay']['best_drawdown_gate_max_drawdown_pct']:.4f}；cross_asset_lgbm_replay 只覆盖到 {CROSS_END}，最佳 20% 回撤候选 return_pct={branch_results['cross_asset_lgbm_replay']['best_drawdown_gate_return_pct']:.4f}。
- 是否晋升：不晋升；未找到同时满足收益超过现有生产策略且 MaxDD < 20% 的候选。
- 风险：这是 diagnostic search，不是 validated PortAnaRecord 晋升卡；跨资产分支覆盖期短于 full-test，不能作为完整目标证据。
- 下一步：需要转向更强的信息面，例如把 2024-2026 Futu 1m intraday T+1 特征接入 qlib store，并做 walk-forward intraday-aware selector，而不是继续扩大日线阈值网格。
- qlib recorder id：{recorder_id}
'''
(CARD / 'conclusion.md').write_text(conclusion, encoding='utf-8')
print(f'recorder_id={recorder_id}')
print(f'passes_both_count={passes_both_count}')
print(json.dumps(branch_results, indent=2, sort_keys=True))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[60585:MainThread](2026-06-23 15:53:25,679) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[60585:MainThread](2026-06-23 15:53:25,681) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[60585:MainThread](2026-06-23 15:53:25,682) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[60585:MainThread](2026-06-23 15:53:25,751) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


...

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[60585:MainThread](2026-06-23 15:53:30,566) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name agentic_alpha_model_replay_search_v1.


[60585:MainThread](2026-06-23 15:53:30,569) INFO - qlib.workflow - [exp.py:258] - Experiment 111386173665924891 starts running ...


[60585:MainThread](2026-06-23 15:53:30,654) INFO - qlib.workflow - [recorder.py:345] - Recorder d6544d1ee4ec4a54b8e9cc847e5b928f starts running under Experiment 111386173665924891 ...


[60585:MainThread](2026-06-23 15:53:30,732) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


recorder_id=d6544d1ee4ec4a54b8e9cc847e5b928f
passes_both_count=0
{
  "cross_asset_lgbm_replay": {
    "best_drawdown_gate_max_drawdown_pct": -19.68776051821107,
    "best_drawdown_gate_return_pct": 175.86828416993777,
    "best_return_max_drawdown_pct": -89.59562655136858,
    "best_return_pct": 3312.7352781525906,
    "comparison_baseline_return_pct": 5134.207948829448,
    "coverage_end": "2026-04-10",
    "coverage_start": "2019-01-02",
    "goal_baseline_return_pct": 6570.0232821043,
    "grid_rows": 1600,
    "passes_both_count": 0
  },
  "production_signal_overlay": {
    "best_drawdown_gate_max_drawdown_pct": -19.668668877630502,
    "best_drawdown_gate_return_pct": 548.7571068933172,
    "best_return_max_drawdown_pct": -62.26319892089549,
    "best_return_pct": 8227.696629793714,
    "comparison_baseline_return_pct": 6570.0232821043,
    "coverage_end": "2026-06-09",
    "coverage_start": "2019-01-02",
    "goal_baseline_return_pct": 6570.0232821043,
    "grid_rows": 420,
    "